## Variance experiment collect from the dataset 

## import all the library

In [ ]:
import asyncio 
import json 
import random 
from pathlib import Path
from dotenv import load_dotenv 
import sys 
import os 
import httpx
import litellm
from src.config import Models

sys.path.insert(0, str(Path().absolute().parent))
load_dotenv()

ROUTER_KEY = os.getenv('ROUTER_KEY')
OPENROUTER_URL = 'https://openrouter.ai/api/v1/chat/completions'

print('Imports OK')

# verify the api keys


In [3]:
Keys = [ 
    'OP_KEY',"AP_KEY","GP_KEY","GQ_KEY","XP_KEY","HF_TOKEN"
]

for  k in Keys:
    os.getenv(k)
    print(f"Keys are ${k} ")


Keys are $OP_KEY 
Keys are $AP_KEY 
Keys are $GP_KEY 
Keys are $GQ_KEY 
Keys are $XP_KEY 
Keys are $HF_TOKEN 


#  Router setup

In [9]:
async def call_model(model: str, prompt: str, temperature: float = 0.7) -> str:
    """Single async call to any model via OpenRouter."""
    headers = {
        'Authorization': f'Bearer {ROUTER_KEY}',
        'Content-Type':  'application/json',
        'HTTP-Referer':  'https://github.com/nexomnis',
    }
    payload = {
        'model':       model,
        'messages':    [{'role': 'user', 'content': prompt}],
        'temperature': temperature,
    }
    async with httpx.AsyncClient(timeout=60) as client:
        r = await client.post(OPENROUTER_URL, headers=headers, json=payload)
        r.raise_for_status()
        return r.json()['choices'][0]['message']['content'] or ''


async def call_models_parallel(prompts: dict[str, tuple[str, str]]) -> dict[str, str]:
    """
    Call multiple models in parallel.
    prompts = {role: (model_id, prompt_text)}
    Returns {role: response_text}
    """
    tasks   = {role: call_model(model, prompt) for role, (model, prompt) in prompts.items()}
    results = await asyncio.gather(*tasks.values(), return_exceptions=True)
    return {
        role: (res if not isinstance(res, Exception) else f'[Error: {res}]')
        for role, res in zip(tasks.keys(), results)
    }

print('Helper functions defined')

Helper functions defined


In [ ]:
async def verify_models():
    for name in Models:
        try:
            resp = await call_model(name, 'reply with the single word: ok')
            print(f'  ✅   ({name[:3]})')
        except Exception as e:
            print(f'  ❌   ({name[:3]}) → {e}')

await verify_models()